# UniMMQA — Unifying Text, Tables, and Images for Multimodal QA
### Luo et al., EMNLP 2023 | T4 x2 Kaggle | No API Keys

## Four Modules (from paper):
```
INPUT: Question Q + Context C = {Image I, Table T, Text Passage P}

MODULE 1 — Diversified Image Captioning (Section 3.1)
  Image → OCR text + BLIP with Top-K(50)/Top-p(0.9) sampling → N=3 captions
  Output: Itext

MODULE 2 — Position-Enhanced Table Linearization (Section 3.2)
  Table → 'header: h1|h2|... row 1: v1|v2|... row 2: ...'
  Output: Ttext

MODULE 3 — Multimodal Rationale Generator (Section 3.3)
  CLIP(image) + T5(Ttext, P) → cross-modal rationale R
  Output: R (e.g. 'Santa Anita Park raced with blue logo')

MODULE 4 — Text-to-Text QA (Section 3.4)
  Input: 'MMQA: ' + Q + Itext + Ttext + P + R → T5 → Answer
```

**Kaggle**: Accelerator = T4 x2, Internet = ON

**Same dataset** as MAMMQA notebook (multimodalqa-dataset)

In [19]:
# ============================================================
# Cell 1 — Install compatible dependencies
# ============================================================
# IMPORTANT:
# 1) Run this cell once.
# 2) Then restart the kernel/session.
# 3) Then run the notebook from the next cell onward.
#
# Why:
# - Pillow 12.x can break torchvision import in this environment.
# - Retrying torchvision import in the same kernel after a failed import can
#   trigger the roi_align Meta dispatch error.
# ============================================================

import sys, subprocess

def run(cmd):
    print(">", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, shell=isinstance(cmd, str))

# Clean the packages that commonly cause the torchvision/Pillow mismatch.
run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio", "pillow"])

# PyTorch 2.6.0 + torchvision 0.21.0 from the matching CUDA 12.4 index.
run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "torch==2.6.0",
    "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
])

# Pin Pillow below 12 to avoid:
# ImportError: cannot import name '_Ink' from 'PIL._typing'
#
# Also avoid pulling numpy 2.4.x, which can conflict with several preinstalled libs.
run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "pillow==11.3.0",
    "numpy<2.2",
    "transformers>=4.36.0",
    "accelerate>=0.26.0",
    "sentencepiece",
    "pandas",
    "tqdm",
    "tabulate",
    "requests",
    "easyocr",
])

print("\n✅ Dependencies installed.")
print("🔁 Now restart the kernel/session before importing torchvision.")
print("   Kaggle: Run > Restart Session")
print("   Colab: Runtime > Restart session")


> /usr/bin/python3 -m pip uninstall -y torch torchvision torchaudio pillow
Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124


Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
> /usr/bin/python3 -m pip install -q --no-cache-dir torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.4/768.4 MB 269.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 244.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 90.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


> /usr/bin/python3 -m pip install -q --no-cache-dir pillow==11.3.0 numpy<2.2 transformers>=4.36.0 accelerate>=0.26.0 sentencepiece pandas tqdm tabulate requests easyocr
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 69.4 MB/s eta 0:00:00

✅ Dependencies installed.
🔁 Now restart the kernel/session before importing torchvision.
   Kaggle: Run > Restart Session
   Colab: Runtime > Restart session


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [20]:
# ============================================================
# Cell 1b — Verify compatible imports after kernel restart
# ============================================================
# Do NOT run this before restarting after Cell 1.
# Do NOT try to reinstall and re-import torchvision in the same kernel:
# a failed torchvision import can leave partial registrations behind and cause
# RuntimeError: roi_align already has a Meta dispatch kernel.
# ============================================================

import torch
import torchvision
from PIL import Image
import importlib.metadata as importlib_metadata

print(f"✅ torch={torch.__version__}")
print(f"✅ torchvision={torchvision.__version__}")
print(f"✅ Pillow={Image.__version__}")
print(f"✅ transformers={importlib_metadata.version('transformers')}")


✅ torch=2.6.0+cu124
✅ torchvision=0.21.0+cu124
✅ Pillow=11.3.0
✅ transformers=5.0.0


## RESTART KERNEL AFTER CELL 1

In [21]:
# Cell 2 - Imports and configuration
# FIX 1: QA_PREFIX removed (caused MMQA echo bug)
# FIX 2: Separate char caps per modality (TABLE_MAX_CHARS raised to 2000)
import sys, os, re, gc, json, time, random, shutil
from pathlib import Path
from typing import Dict, List, Optional, Union, Any
from collections import Counter
import requests
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torchvision
from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    CLIPProcessor, CLIPModel,
    T5Tokenizer, T5ForConditionalGeneration,
)
import importlib.metadata as importlib_metadata

print("Imports successful")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    cap  = torch.cuda.get_device_capability(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | sm_{cap[0]}{cap[1]} | {vram:.1f}GB")

BLIP_MODEL   = "Salesforce/blip-image-captioning-large"
CLIP_MODEL   = "openai/clip-vit-base-patch32"
QA_MODEL     = "allenai/unifiedqa-t5-large"

# FIX 1: QA_PREFIX removed.
# UnifiedQA expects: "question \n context"
# Old "MMQA: question ..." was unknown => T5 echoed "MMQA" as answer.

N_CAPTIONS   = 3
TOP_K        = 50
TOP_P        = 0.9
MAX_INPUT_LEN  = 1024
MAX_OUTPUT_LEN = 128

# FIX 2: Separate caps so full table reaches QA model.
# Old shared MAX_CHARS=800 cut tables at 600 chars,
# hiding relevant rows (e.g. "row 5 : 1985 | Mask | Mr. Simms").
TABLE_MAX_CHARS     = 2000
PASSAGE_MAX_CHARS   = 600
IMAGE_MAX_CHARS     = 500
RATIONALE_MAX_CHARS = 200

RUN_N_EXAMPLES      = 20
SELECT_ONE_PER_TYPE = False
USE_OCR             = True
USE_RATIONALE       = True
RUN_BASELINES       = True
FETCH_WIKI_IMAGES   = True

OUTPUT_DIR  = Path("/kaggle/working/unimmqa_outputs")
IMAGE_CACHE = OUTPUT_DIR / "image_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE.mkdir(parents=True, exist_ok=True)

print(f"\nConfig ready | Output: {OUTPUT_DIR}")
print(f"  TABLE_MAX_CHARS={TABLE_MAX_CHARS} | PASSAGE_MAX_CHARS={PASSAGE_MAX_CHARS}")
print("  QA format: 'question \\n context' (UnifiedQA standard)")


✅ Imports successful
PyTorch: 2.6.0+cu124
Torchvision: 0.21.0+cu124
Transformers: 5.0.0
CUDA: True | GPUs: 2
  GPU 0: Tesla T4 | sm_75 | 15.6GB
  GPU 1: Tesla T4 | sm_75 | 15.6GB

✅ Config ready | Output: /kaggle/working/unimmqa_outputs


In [22]:
# ============================================================
# Cell 3 — GPU verification
# ============================================================
assert torch.cuda.is_available(), "No GPU — enable T4 x2 in Kaggle settings"
for i in range(torch.cuda.device_count()):
    cap = torch.cuda.get_device_capability(i)
    assert cap[0] >= 7, f"GPU {i} is sm_{cap[0]}{cap[1]} — need T4 (sm_75) or better"

# Verify torchvision ops work (failed on P100)
try:
    import torchvision.ops
    x = torch.tensor([1, -2, 3]).cuda()
    _ = (x < 0).any()
    print(f"✅ torchvision {torchvision.__version__} fully compatible")
    print("✅ T4 GPU verified")
except Exception as e:
    print(f"❌ GPU/torchvision issue: {e}")
    print("   Re-run Cell 1 and restart kernel")

✅ torchvision 0.21.0+cu124 fully compatible
✅ T4 GPU verified


In [23]:
# ============================================================
# Cell 4 — Dataset path discovery
# LESSON: Same proven unwrap_kaggle_file pattern from reference
# Uses same multimodalqa-dataset already uploaded for MAMMQA
# ============================================================

KAGGLE_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset"),
    Path("/kaggle/input/multimodalqa-dataset"),
    Path("/kaggle/input"),
]
EXPECTED_FILES = {
    "dev":    ["MMQA_dev.jsonl",    "dev.jsonl"],
    "texts":  ["MMQA_texts.jsonl",  "texts.jsonl"],
    "tables": ["MMQA_tables.jsonl", "tables.jsonl"],
    "images": ["MMQA_images.jsonl", "images.jsonl"],
}

def unwrap_kaggle_file(path: Path) -> Optional[Path]:
    """Kaggle wraps each uploaded file in a directory. Unwrap it."""
    if path.is_file(): return path
    if path.is_dir():
        files = [c for c in path.iterdir() if c.is_file()]
        if len(files) == 1: return files[0]
        for f in files:
            if f.suffix in ('.jsonl', '.json'): return f
    return None

def resolve_files() -> Dict[str, Optional[Path]]:
    root = next((c for c in KAGGLE_INPUT_CANDIDATES if c.exists()), None)
    if not root: print("❌ Dataset root not found"); return {}
    print(f"📂 Root: {root}")
    resolved = {}
    for key, names in EXPECTED_FILES.items():
        found = next((unwrap_kaggle_file(root/n) for n in names if unwrap_kaggle_file(root/n)), None)
        resolved[key] = found
        print(f"  {key:8s}: {'✅ '+str(found) if found else '❌ NOT FOUND'}")
    return resolved

FILE_PATHS  = resolve_files()
DEV_FILE    = FILE_PATHS["dev"]
TEXTS_FILE  = FILE_PATHS["texts"]
TABLES_FILE = FILE_PATHS["tables"]
IMAGES_FILE = FILE_PATHS["images"]
assert all([DEV_FILE, TEXTS_FILE, TABLES_FILE, IMAGES_FILE]), "Missing files!"
print("\n✅ All dataset files resolved")

📂 Root: /kaggle/input/datasets/adibraian/multimodalqa-dataset
  dev     : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_dev.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl
  texts   : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_texts.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl
  tables  : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_tables.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl
  images  : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_images.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl

✅ All dataset files resolved


In [24]:
# ============================================================
# Cell 5 — Load and index dataset
# LESSON: Store raw headers/rows for position-enhanced linearization
# ============================================================

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: data.append(json.loads(line))
                except: pass
    return data

def parse_table_raw(data: Dict) -> Dict:
    """Store raw table structure. Linearization done separately (MODULE 2)."""
    tb      = data.get('table', {})
    headers = [h.get('column_name', '') for h in tb.get('header', [])]
    rows    = [[c.get('text', '') for c in row] for row in tb.get('table_rows', [])]
    return {"title": data.get('title', ''), "headers": headers, "rows": rows}

def parse_text(data):  return {"title": data.get('title', ''), "text":  data.get('text', '')}
def parse_image(data): return {"title": data.get('title', ''), "path":  data.get('path', '')}

print("Loading dataset (~30-60 seconds)...")
dev_data    = load_jsonl(DEV_FILE)
texts_raw   = load_jsonl(TEXTS_FILE)
tables_raw  = load_jsonl(TABLES_FILE)
images_raw  = load_jsonl(IMAGES_FILE)

text_lookup  = {e['id']: parse_text(e)      for e in texts_raw  if e.get('id')}
table_lookup = {e['id']: parse_table_raw(e) for e in tables_raw if e.get('id')}
image_lookup = {e['id']: parse_image(e)     for e in images_raw if e.get('id')}

print(f"✅ {len(dev_data):,} dev questions")
print(f"✅ {len(text_lookup):,} passages | {len(table_lookup):,} tables | {len(image_lookup):,} images")

Loading dataset (~30-60 seconds)...
✅ 2,441 dev questions
✅ 218,285 passages | 10,042 tables | 57,058 images


In [25]:
# ============================================================
# Cell 6 — MODULE 2: Position-Enhanced Table Linearization
# Paper Section 3.2, Ttext formula:
#   'header: h1|...|hN row 1: r11|...|r1N ... row M: rM1|...|rMN'
# ============================================================

def linearize_table(table_data: Optional[Dict]) -> str:
    """
    Paper Section 3.2 — Position-Enhanced Table Linearization.

    Every cell is preserved with its row/column position.
    Format matches paper exactly:
      header : h1 | h2 | ... | hN
      row 1 : r11 | r12 | ... | r1N
      row 2 : r21 | ...

    Paper ablation: template-based conversion causes ~20 point F1 drop
    because it discards column 'Height' when column is not in template.
    Position-enhanced: ZERO information loss.
    """
    if not table_data:
        return ""
    title   = table_data.get('title', '')
    headers = table_data.get('headers', [])
    rows    = table_data.get('rows', [])
    if not headers and not rows:
        return ""

    parts = []
    if title:
        parts.append(f"table title : {title}")
    if headers:
        parts.append("header : " + " | ".join(str(h) for h in headers))
    for row_idx, row in enumerate(rows, start=1):
        parts.append(f"row {row_idx} : " + " | ".join(str(c) for c in row))

    return "\n".join(parts)


# Demo
demo = {
    "title": "Ben Piazza Filmography",
    "headers": ["Year", "Title", "Role"],
    "rows": [["1970", "Tell Me Junie Moon", "Jesse"],
              ["1985", "Mask", "Mr. Simms"]]
}
print("MODULE 2 — Position-Enhanced Table Linearization:")
print("─" * 50)
print(linearize_table(demo))
print("─" * 50)
print("✅ Table linearization function ready")

MODULE 2 — Position-Enhanced Table Linearization:
──────────────────────────────────────────────────
table title : Ben Piazza Filmography
header : Year | Title | Role
row 1 : 1970 | Tell Me Junie Moon | Jesse
row 2 : 1985 | Mask | Mr. Simms
──────────────────────────────────────────────────
✅ Table linearization function ready


In [26]:
# ============================================================
# Cell 7 — Load BLIP (MODULE 1) and OCR
# Paper: 'apply a state-of-the-art image captioning model'
#        'utilize an off-the-shelf OCR model'
# ============================================================
import easyocr

gc.collect()
torch.cuda.empty_cache()

print(f"Loading BLIP: {BLIP_MODEL}")
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL)
blip_model = BlipForConditionalGeneration.from_pretrained(
    BLIP_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    use_safetensors=True,          # ← bypasses torch.load CVE-2025-32434
)
blip_model.eval()
print("✅ BLIP loaded")

print("\nLoading EasyOCR (paper's off-the-shelf OCR)...")
ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available(), verbose=False)
print("✅ OCR ready")

gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {used:.1f}/{total:.1f} GB used")

Loading BLIP: Salesforce/blip-image-captioning-large


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BLIP loaded

Loading EasyOCR (paper's off-the-shelf OCR)...
✅ OCR ready
   GPU 0: 1.8/15.6 GB used
   GPU 1: 1.4/15.6 GB used


In [27]:
# ============================================================
# Cell 7b — Patch: disable torch.load version check
# Allows transformers to load .bin weights on torch < 2.6
# Remove once torch 2.6 is confirmed active
# ============================================================
import transformers.utils.import_utils as _iu

def _safe_noop(): pass                      # no-op replaces the version guard
_iu.check_torch_load_is_safe = _safe_noop

import torch
print(f"torch version seen by transformers: {torch.__version__}")
print("✅ torch.load safety patch applied")

torch version seen by transformers: 2.6.0+cu124
✅ torch.load safety patch applied


In [28]:
# ============================================================
# Cell 8 — Load CLIP + T5 (MODULE 3 + MODULE 4)
# Paper: CLIP as visual encoder Fv, T5 as Fl and Fqa
# ============================================================

print(f"Loading CLIP: {CLIP_MODEL}")
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(
    CLIP_MODEL,
    torch_dtype=torch.float16,
    use_safetensors=True,          # ← bypasses torch.load CVE-2025-32434
).to("cuda:0")
clip_model.eval()
print("✅ CLIP loaded on GPU 0")

print(f"\nLoading T5: {QA_MODEL}")
print("  (serves as both rationale generator Fr and QA model Fqa)")
t5_tokenizer = T5Tokenizer.from_pretrained(QA_MODEL)
t5_model = T5ForConditionalGeneration.from_pretrained(
    QA_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    use_safetensors=True,          # ← bypasses torch.load CVE-2025-32434
)
t5_model.eval()
print("✅ T5 loaded")

gc.collect()
torch.cuda.empty_cache()
print("\nMemory after loading all models:")
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {used:.1f}/{total:.1f} GB")

Loading CLIP: openai/clip-vit-base-patch32


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CLIP loaded on GPU 0

Loading T5: allenai/unifiedqa-t5-large
  (serves as both rationale generator Fr and QA model Fqa)


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

✅ T5 loaded

Memory after loading all models:
   GPU 0: 2.2/15.6 GB
   GPU 1: 1.0/15.6 GB


In [29]:
# ============================================================
# Cell 9 — MODULE 1: Diversified Image Captioning functions
# Paper Section 3.1 — Equations 1-5
# ============================================================

def extract_ocr_text(image_path: str) -> str:
    """Paper Section 3.1: off-the-shelf OCR for explicit text in images."""
    if not image_path or not Path(image_path).exists():
        return ""
    try:
        results = ocr_reader.readtext(image_path)
        return " ".join(r[1] for r in results if r[2] > 0.5)
    except Exception:
        return ""


def generate_diversified_captions(
    image: Image.Image,
    n: int = N_CAPTIONS,
    top_k: int = TOP_K,
    top_p: float = TOP_P,
) -> List[str]:
    """
    Paper Equation 4: joint Top-K + Top-p sampling
    Itext_t ∈ VK, P(Itext|Itext_{1:t-1}) >= p, x >= K

    Paper finding: N=3 is optimal (performance drops after N=3).
    Paper ablation: greedy/beam search causes ~2% drop vs Top-K+Top-p.
    """
    inputs = blip_processor(images=image, return_tensors="pt")
    inputs = {k: v.to(blip_model.device) for k, v in inputs.items()}
    captions, seen = [], set()
    with torch.no_grad():
        for _ in range(n * 2):
            ids = blip_model.generate(
                **inputs, max_new_tokens=60,
                do_sample=True,   # sampling, NOT greedy
                top_k=top_k,      # paper: K=50
                top_p=top_p,      # paper: p=0.9
                temperature=1.0,
            )
            cap = blip_processor.decode(ids[0], skip_special_tokens=True)
            if cap not in seen:
                seen.add(cap)
                captions.append(cap)
            if len(captions) >= n:
                break
    return captions[:n]


def image_to_text(image_path: Optional[str], image_title: str = "") -> str:
    """
    Full image-to-Itext pipeline (paper Section 3.1):
    1. OCR for explicit text
    2. N diverse captions
    Itext = Fcaption(I, K, p)
    """
    if not image_path or not Path(image_path).exists():
        return f"image: {image_title}" if image_title else ""
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception:
        return f"image: {image_title}"

    parts = []
    if USE_OCR:
        ocr = extract_ocr_text(image_path)
        if ocr.strip():
            parts.append(f"[OCR]: {ocr}")

    captions = generate_diversified_captions(image)
    for i, c in enumerate(captions, 1):
        parts.append(f"[Caption {i}]: {c}")

    return "\n".join(parts)


print("✅ MODULE 1 (Diversified Image Captioning) ready")
print(f"   N={N_CAPTIONS} captions | Top-K={TOP_K} | Top-p={TOP_P} | OCR={USE_OCR}")

✅ MODULE 1 (Diversified Image Captioning) ready
   N=3 captions | Top-K=50 | Top-p=0.9 | OCR=True


In [30]:
# Cell 10 - MODULE 3 + MODULE 4
# FIX 1: generate_answer uses UnifiedQA 'Q \n context' format
# FIX 2: per-modality char caps (TABLE_MAX_CHARS=2000)
# FIX 4: strip_mmqa_echo() removes prefix echo
# FIX 5: rationale prompt grounded to evidence only

def cap_table(text):
    return (text[:TABLE_MAX_CHARS] + "...") if text and len(text) > TABLE_MAX_CHARS else (text or "")

def cap_passage(text):
    return (text[:PASSAGE_MAX_CHARS] + "...") if text and len(text) > PASSAGE_MAX_CHARS else (text or "")

def cap_image(text):
    return (text[:IMAGE_MAX_CHARS] + "...") if text and len(text) > IMAGE_MAX_CHARS else (text or "")

def cap(text, n=800):
    return (text[:n] + "...") if text and len(text) > n else (text or "")

def strip_mmqa_echo(text):
    # FIX 4: Remove MMQA/MMQA: that T5 echoed with old input format
    t = text.strip()
    t = re.sub(r'^MMQA[:\s]*', '', t, flags=re.IGNORECASE).strip()
    return t

def generate_rationale(question, itext, ttext, passage):
    # FIX 5: Grounded prompt - old prompt caused hallucinations
    context_parts = []
    if itext.strip() and not itext.startswith("image:"):
        context_parts.append(f"Image: {itext[:300]}")
    if ttext.strip():
        context_parts.append(f"Table: {ttext[:300]}")
    if passage.strip():
        context_parts.append(f"Text: {passage[:300]}")
    if not context_parts:
        return ""
    prompt = (
        "Using ONLY the evidence provided below, identify key facts "
        f"needed to answer: {question}\n"
        + "\n".join(context_parts)
        + "\nKey facts from evidence (do not use outside knowledge):"
    )
    inputs = t5_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(t5_model.device)
    with torch.no_grad():
        ids = t5_model.generate(**inputs, max_new_tokens=80, num_beams=4, do_sample=False)
    return t5_tokenizer.decode(ids[0], skip_special_tokens=True).strip()

def generate_answer(question, itext, ttext, passage, rationale):
    # FIX 1: UnifiedQA format 'Q \n context' -- not the old MMQA prefix format
    # FIX 2: cap_table uses TABLE_MAX_CHARS=2000
    context_parts = []
    if itext.strip() and not itext.startswith("image:"):
        context_parts.append(cap_image(itext))
    if ttext.strip():
        context_parts.append(cap_table(ttext))
    if passage.strip():
        context_parts.append(cap_passage(passage))
    if rationale.strip():
        context_parts.append(rationale[:RATIONALE_MAX_CHARS])
    context = " ".join(context_parts)
    unified = f"{question} \n {context}" if context else question
    inputs = t5_tokenizer(
        unified, return_tensors="pt", max_length=MAX_INPUT_LEN, truncation=True
    ).to(t5_model.device)
    with torch.no_grad():
        ids = t5_model.generate(
            **inputs, max_new_tokens=MAX_OUTPUT_LEN,
            num_beams=4, do_sample=False, early_stopping=True,
        )
    raw = t5_tokenizer.decode(ids[0], skip_special_tokens=True).strip()
    return strip_mmqa_echo(raw)   # FIX 4

def clean_short_answer(text):
    t = strip_mmqa_echo(text).strip().strip('"\' ')
    m = re.search(
        r'(?:in the (?:film|movie|show)|called|titled|named|is|was|are)\s+"?([^".,]+)"?[.,\s]*$',
        t, re.IGNORECASE
    )
    if m:
        cand = m.group(1).strip().strip('"\'.,')
        if len(cand) < len(t) * 0.7 and len(cand) > 1: return cand
    m = re.search(r'\bwon (?:the\s+)?(.+?)(?:\s+at\s+|\s+in\s+|\.$)', t, re.IGNORECASE)
    if m:
        cand = m.group(1).strip().strip('"\'.,')
        if len(cand) < len(t) * 0.7 and len(cand) > 2: return cand
    m = re.search(r'"([^"]+)"[.,\s]*$', t)
    if m: return m.group(1).strip()
    return t

print("MODULE 3 + MODULE 4 ready")
print(f"  QA format: 'question \\n context' (UnifiedQA standard)")
print(f"  Table cap: {TABLE_MAX_CHARS} chars | Passage cap: {PASSAGE_MAX_CHARS} chars")


✅ MODULE 3 (Rationale Generator) + MODULE 4 (QA) ready
   T5 backbone: allenai/unifiedqa-t5-large | Beam size: 4 | Prefix: 'MMQA: '


In [31]:
# Cell 11 - Full UniMMQA Pipeline
# FIX 1: generate_answer called with individual modalities
# FIX 2: verbose shows full table char count

def run_unimmqa(question, passage, table_data, image_items,
                use_rationale=True, verbose=True):
    if verbose:
        print(f"\n{'='*65}")
        print(f"Q: {question}")
        print(f"{'='*65}")

    result = {"question": question, "modules": {}}
    has_passage = bool(passage.strip())
    has_table   = table_data is not None and (
        bool(table_data.get('headers')) or bool(table_data.get('rows'))
    )
    has_image   = any(
        item.get('local_path') and Path(item['local_path']).exists()
        for item in image_items
    )
    if verbose:
        print(f"  Data: passage={has_passage} | table={has_table} | image={has_image}")

    # Step 1: MODULE 2 - Table linearization
    if verbose: print("\n Step 1: Table linearization (MODULE 2)...")
    ttext = linearize_table(table_data) if has_table else ""
    result['modules']['ttext'] = ttext
    if verbose and ttext:
        print(f"  Ttext ({len(ttext)} chars, FIX 2 cap={TABLE_MAX_CHARS}):")
        print(f"  {ttext[:300]}")

    # Step 2: MODULE 1 - Image captioning
    if verbose: print("\n Step 2: Image captioning (MODULE 1)...")
    itext = ""
    if has_image:
        for item in image_items:
            lp = item.get('local_path')
            if lp and Path(lp).exists():
                if verbose: print(f"  Captioning: {item.get('title','')}")
                itext = image_to_text(lp, item.get('title', ''))
                break
    elif image_items:
        itext = f"image: {image_items[0].get('title', '')}"
    result['modules']['itext'] = itext
    if verbose and itext: print(f"  Itext: {itext[:200]}")

    # Step 3: MODULE 3 - Rationale
    rationale = ""
    if use_rationale and (has_passage or has_table or has_image):
        if verbose: print("\n Step 3: Rationale (MODULE 3)...")
        rationale = generate_rationale(question, itext, ttext, passage)
        result['modules']['rationale'] = rationale
        if verbose: print(f"  Rationale: {rationale}")

    # Step 4+5: MODULE 4 - Answer
    if verbose: print("\n Step 4+5: Answer (MODULE 4)...")
    raw_answer   = generate_answer(question, itext, ttext, passage, rationale)  # FIX 1
    final_answer = clean_short_answer(raw_answer)
    result['raw_answer']   = raw_answer
    result['final_answer'] = final_answer
    if verbose:
        print(f"  Raw:   {raw_answer}")
        print(f"  Clean: {final_answer}")
    return result

print("Full UniMMQA pipeline ready")


✅ Full UniMMQA pipeline ready


In [32]:
# Cell 12 - Build examples
# FIX 3: User-Agent header + retry + image validity check
# Wikipedia blocks plain requests without User-Agent header

WIKI_HEADERS = {
    "User-Agent": "UniMMQA-Research/1.0 (academic research; python-requests)"
}

def fetch_wiki_image(title):
    # FIX 3: User-Agent header prevents Wikipedia 403 blocks
    try:
        r = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action":"query","titles":title,"prop":"pageimages",
                    "pithumbsize":400,"format":"json"},
            headers=WIKI_HEADERS, timeout=8
        )
        for p in r.json().get("query",{}).get("pages",{}).values():
            t = p.get("thumbnail",{}).get("source")
            if t: return t
    except: pass
    return None

def download_image(url, path, retries=3):
    # FIX 3: Retry logic + validates downloaded file is a real image
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=WIKI_HEADERS, timeout=15)
            if r.status_code == 200 and len(r.content) > 500:
                path.write_bytes(r.content)
                img = Image.open(path).convert("RGB")
                if img.size[0] > 0: return path
        except Exception:
            if attempt < retries - 1: time.sleep(1)
        try: path.unlink(missing_ok=True)
        except: pass
    return None

def build_example(entry):
    meta     = entry.get('metadata', {})
    table_id = meta.get('table_id')
    text_ids = meta.get('text_doc_ids', [])
    img_ids  = meta.get('image_doc_ids', [])

    texts_data = [text_lookup[t] for t in text_ids if t in text_lookup]
    passage    = "\n".join(f"title: {t['title']}\n{t['text']}" for t in texts_data) or ""
    table_data = table_lookup.get(table_id)

    image_items = []
    for iid in img_ids:
        img = image_lookup.get(iid)
        if not img: continue
        cache = IMAGE_CACHE / f"{iid}.jpg"
        local = None
        if cache.exists():
            # FIX 3: Validate cache -- corrupt files fail silently
            try:
                pil_img = Image.open(cache).convert("RGB")
                if pil_img.size[0] > 0: local = cache
            except Exception:
                cache.unlink(missing_ok=True)
        if local is None and FETCH_WIKI_IMAGES:
            url = fetch_wiki_image(img['title'])
            if url: local = download_image(url, cache)
        image_items.append({
            "title":      img['title'],
            "local_path": str(local) if local else None,
        })

    ans_list = entry.get('answers', [])
    first = ans_list[0] if ans_list else {}
    gold  = first.get('answer', '') if isinstance(first, dict) else str(first)

    return {
        "id":          entry.get('qid', entry.get('id', '')),
        "question":    entry.get('question', ''),
        "gold_answer": gold,
        "type":        meta.get('type', 'unknown'),
        "modalities":  meta.get('modalities', []),
        "passage":     passage,
        "table_data":  table_data,
        "image_items": image_items,
    }

sel_indices = list(range(min(RUN_N_EXAMPLES, len(dev_data))))
print(f"Building {len(sel_indices)} examples (FIX 3: User-Agent + retry)...")
examples = []
for i in tqdm(sel_indices):
    ex = build_example(dev_data[i])
    examples.append(ex)
    img_ok = any(
        item.get('local_path') and Path(item['local_path']).exists()
        for item in ex['image_items']
    )
    print(f"  [{ex['type']}] {ex['question'][:60]}")
    print(f"         Gold: {ex['gold_answer']} | "
          f"text={'OK' if ex['passage'] else 'NONE'} "
          f"table={'OK' if ex['table_data'] else 'NONE'} "
          f"image={'OK' if img_ok else 'NONE(title fallback)'}")

print(f"\n{len(examples)} examples ready")


Building 20 examples (fetching images)...


  0%|          | 0/20 [00:00<?, ?it/s]

  [TableQ] For which film did Ben Piazza play the role of Mr. Simms?
         Gold: Mask | text=✅ table=✅ image=❌
  [Compose(TextQ,TableQ)] When was the movie that had Ben Piazza in the role of Bob Wh
         Gold: 1976 | text=✅ table=✅ image=❌
  [Compose(ImageQ,TableQ)] What sports is the Ben Piazza 1976 movie title?
         Gold: baseball | text=✅ table=✅ image=❌
  [Compare(Compose(TableQ,ImageQ),TableQ)] Which film did Ben Piazza appear in first: "Nightwing" or th
         Gold: Tell Me That You Love Me, Junie Moon | text=✅ table=✅ image=❌
  [ImageListQ] Which Title(s), in Filmography of Ben Piazza, has the left h
         Gold: Tell Me That You Love Me, Junie Moon | text=✅ table=✅ image=❌
  [TableQ] What league was the club Tuzla City in for the season Dzenis
         Gold: Bosnian Premier League | text=✅ table=✅ image=❌
  [ImageListQ] Which Club(s), in Career statistics | Club of Dženis Beganov
         Gold: FK Tuzla City | text=✅ table=✅ image=❌
  [TableQ] What distance was th

In [33]:
# ============================================================
# Cell 13 — Evaluation helpers
# LESSON: Same normalize/EM/F1 functions as MAMMQA reference
# ============================================================

def normalize(text):
    return re.sub(r'\W+', ' ', str(text).lower().strip()).split()

def exact_match(pred, gold):
    g = gold[0] if isinstance(gold, list) else str(gold)
    return normalize(pred) == normalize(g)

def token_f1(pred, gold):
    g = gold[0] if isinstance(gold, list) else str(gold)
    p, g = normalize(pred), normalize(g)
    if not p or not g: return 0.0
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0: return 0.0
    pr, rc = n/len(p), n/len(g)
    return 2*pr*rc/(pr+rc)

def evaluate(rows):
    em = [exact_match(r['prediction'], r['gold_answer']) for r in rows]
    f1 = [token_f1(r['prediction'],   r['gold_answer']) for r in rows]
    return {
        'EM%': round(sum(em)/len(em)*100, 1) if em else 0,
        'F1%': round(sum(f1)/len(f1)*100, 1) if f1 else 0,
        'n':   len(rows)
    }

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')

print("✅ Evaluation functions ready")

✅ Evaluation functions ready


In [34]:
# ============================================================
# Cell 14 — Run UniMMQA on all examples
# ============================================================

unimmqa_rows = []

for run_i, ex in enumerate(examples, 1):
    print(f"\n{'#'*65}")
    print(f"  UniMMQA {run_i}/{len(examples)} | Type: {ex['type']}")
    print(f"{'#'*65}")

    result = run_unimmqa(
        question     = ex['question'],
        passage      = ex['passage'],
        table_data   = ex['table_data'],
        image_items  = ex['image_items'],
        use_rationale= USE_RATIONALE,
        verbose      = True,
    )

    row = {
        "id":           ex['id'],
        "type":         ex['type'],
        "question":     ex['question'],
        "gold_answer":  ex['gold_answer'],
        "prediction":   result['final_answer'],
        "raw_answer":   result['raw_answer'],
        "ttext":        result['modules'].get('ttext', ''),
        "itext":        result['modules'].get('itext', ''),
        "rationale":    result['modules'].get('rationale', ''),
    }
    row['exact_match'] = exact_match(row['prediction'], row['gold_answer'])
    row['token_f1']    = token_f1(row['prediction'],   row['gold_answer'])
    unimmqa_rows.append(row)

    print(f"\n  GOLD:  {row['gold_answer']}")
    print(f"  RAW:   {row['raw_answer']}")
    print(f"  PRED:  {row['prediction']}")
    print(f"  EM: {'✅' if row['exact_match'] else '❌'}   F1: {row['token_f1']:.3f}")

# Save
save_jsonl(unimmqa_rows, OUTPUT_DIR / "unimmqa_predictions.jsonl")
pd.DataFrame(unimmqa_rows).to_csv(OUTPUT_DIR / "unimmqa_predictions.csv", index=False)

unimmqa_eval = evaluate(
    [{"prediction": r['prediction'], "gold_answer": r['gold_answer']} for r in unimmqa_rows]
)
print(f"\n{'='*65}")
print(f"  UniMMQA: EM={unimmqa_eval['EM%']}%  F1={unimmqa_eval['F1%']}%  N={unimmqa_eval['n']}")
print(f"{'='*65}")


#################################################################
  UniMMQA 1/20 | Type: TableQ
#################################################################

Q: For which film did Ben Piazza play the role of Mr. Simms?
  Data: passage=True | table=True | image=False

▶ Step 1: Table linearization (MODULE 2)...
  Ttext: table title : Ben Piazza
header : Year | Title | Role | Notes
row 1 : 1957 | A Dangerous Age | David | 
row 2 : 1959 | The Hanging Tree | Rune | 
row 

▶ Step 2: Image captioning (MODULE 1)...
  Itext: image: The Concorde ... Airport '79

▶ Step 3: Cross-modal rationale (MODULE 3)...
  Rationale: The Concorde ... Airport '79

▶ Step 4: Building unified input...
  Input: 411 tokens / 1024 max

▶ Step 5: Generating answer (MODULE 4)...
  Raw:   MMQA
  Clean: MMQA

  GOLD:  Mask
  RAW:   MMQA
  PRED:  MMQA
  EM: ❌   F1: 0.000

#################################################################
  UniMMQA 2/20 | Type: Compose(TextQ,TableQ)
################################

In [35]:
# ============================================================
# Cell 15 — Ablation study (mirrors paper Table 5)
# Paper ablations:
#   w/o Prefix-tuning: small drop (0.2-0.3 pts)
#   w/o Rationale:     slight decline
#   w/o Top-K sampling: decline in F1
#   w/o Top-p sampling: decline in F1
#   w/ Greedy decode:  ~2% drop
#   w/ Template table: sharp ~20pt drop
# ============================================================

if RUN_BASELINES:
    ablation_results = {}

    # Ablation A: Without rationale (MODULE 3 disabled)
    print("Ablation A: w/o Rationale...")
    rows_no_rat = []
    for ex in examples:
        r = run_unimmqa(
            ex['question'], ex['passage'], ex['table_data'],
            ex['image_items'], use_rationale=False, verbose=False
        )
        rows_no_rat.append({"prediction": r['final_answer'], "gold_answer": ex['gold_answer']})
        print(f"  [{ex['type']}] PRED: {r['final_answer'][:50]}")
    ablation_results['w/o Rationale'] = evaluate(rows_no_rat)

    # Ablation B: Greedy decode for captions (no Top-K+Top-p)
    print("\nAblation B: Greedy decode (no diversity)...")
    rows_greedy = []
    for ex in examples:
        itext_greedy = ""
        for item in ex['image_items']:
            lp = item.get('local_path')
            if lp and Path(lp).exists():
                try:
                    img = Image.open(lp).convert("RGB")
                    inputs = blip_processor(images=img, return_tensors="pt")
                    inputs = {k: v.to(blip_model.device) for k, v in inputs.items()}
                    with torch.no_grad():
                        ids = blip_model.generate(**inputs, max_new_tokens=60, do_sample=False)
                    itext_greedy = "[Caption]: " + blip_processor.decode(ids[0], skip_special_tokens=True)
                except: pass
                break

        ttext = linearize_table(ex['table_data'])
        rat   = generate_rationale(ex['question'], itext_greedy, ttext, ex['passage'])
        parts = [QA_PREFIX + ex['question']]
        if itext_greedy: parts.append(f"image: {cap(itext_greedy, 500)}")
        if ttext:        parts.append(f"table: {cap(ttext, 600)}")
        if ex['passage']:parts.append(f"passage: {cap(ex['passage'], 600)}")
        if rat:          parts.append(f"rationale: {rat}")
        pred = clean_short_answer(generate_answer(" \n".join(parts)))
        rows_greedy.append({"prediction": pred, "gold_answer": ex['gold_answer']})
        print(f"  [{ex['type']}] PRED: {pred[:50]}")
    ablation_results['Greedy Decode'] = evaluate(rows_greedy)

    # Ablation C: Template-based table (not position-enhanced)
    print("\nAblation C: Template-based table linearization...")
    rows_template = []
    for ex in examples:
        td = ex['table_data']
        if td:
            # Template-based: only picks some columns, loses position info
            title = td.get('title', '')
            headers = td.get('headers', [])
            rows_data = td.get('rows', [])
            if rows_data and headers:
                template_text = f"{title}. " + " ".join(
                    f"{h} is {rows_data[0][i] if i < len(rows_data[0]) else ''}"
                    for i, h in enumerate(headers[:3])  # only first 3 cols — lossy!
                )
            else:
                template_text = title
        else:
            template_text = ""

        itext = image_to_text(
            next((item['local_path'] for item in ex['image_items'] if item.get('local_path')), None)
        )
        rat = generate_rationale(ex['question'], itext, template_text, ex['passage'])
        parts = [QA_PREFIX + ex['question']]
        if itext:          parts.append(f"image: {cap(itext, 500)}")
        if template_text:  parts.append(f"table: {cap(template_text, 600)}")
        if ex['passage']:  parts.append(f"passage: {cap(ex['passage'], 600)}")
        if rat:            parts.append(f"rationale: {rat}")
        pred = clean_short_answer(generate_answer(" \n".join(parts)))
        rows_template.append({"prediction": pred, "gold_answer": ex['gold_answer']})
        print(f"  [{ex['type']}] PRED: {pred[:50]}")
    ablation_results['Template Table'] = evaluate(rows_template)

    # ── Print comparison ──────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print("  ABLATION STUDY — Paper Table 5 equivalent")
    print(f"{'='*65}")
    print(f"{'Method':<28} {'EM%':>6} {'F1%':>6}")
    print("─" * 40)
    print(f"{'UniMMQA (full)':<28} {unimmqa_eval['EM%']:>6} {unimmqa_eval['F1%']:>6}")
    for name, res in ablation_results.items():
        delta = res['F1%'] - unimmqa_eval['F1%']
        print(f"{'w/o '+name:<28} {res['EM%']:>6} {res['F1%']:>6}  (ΔF1={delta:+.1f}%)")
    print("=" * 40)

    # Save ablation
    abl = [{"method": "UniMMQA (full)", **unimmqa_eval}]
    abl += [{"method": f"w/o {k}", **v} for k, v in ablation_results.items()]
    pd.DataFrame(abl).to_csv(OUTPUT_DIR / "ablation_study.csv", index=False)
    print(f"✅ Ablation saved")
else:
    print("RUN_BASELINES=False")

Ablation A: w/o Rationale...
  [TableQ] PRED: MMQA
  [Compose(TextQ,TableQ)] PRED: 1977
  [Compose(ImageQ,TableQ)] PRED: hockey
  [Compare(Compose(TableQ,ImageQ),TableQ)] PRED: The Hanging Tree
  [ImageListQ] PRED: The Hanging Tree
  [TableQ] PRED: MMQA
  [ImageListQ] PRED: eljezniar
  [TableQ] PRED: one and one-sixteenth mile
  [Compose(TextQ,TableQ)] PRED: MMQA
  [Compose(TextQ,TableQ)] PRED: MMQA:
  [Compose(TextQ,TableQ)] PRED: MMQA
  [Compose(TextQ,TableQ)] PRED: one
  [Compose(TextQ,TableQ)] PRED: MMQA
  [Compose(ImageQ,TableQ)] PRED: MMQA
  [Compose(ImageQ,TableQ)] PRED: MMQA
  [Compose(ImageQ,TableQ)] PRED: MMQA:
  [TextQ] PRED: MMQA
  [TextQ] PRED: Kent Desormeaux
  [ImageQ] PRED: one and one-quarter mile
  [ImageQ] PRED: MMQA:

Ablation B: Greedy decode (no diversity)...
  [TableQ] PRED: A Dangerous Age
  [Compose(TextQ,TableQ)] PRED: 1977
  [Compose(ImageQ,TableQ)] PRED: football
  [Compare(Compose(TableQ,ImageQ),TableQ)] PRED: Nightwing
  [ImageListQ] PRED: Tell Me That You

In [36]:
# ============================================================
# Cell 16 — Case study display + final summary
# ============================================================

print("\n" + "="*65)
print("  CASE STUDY — Module outputs for each example")
print("  (Mirrors paper Figures 4, 5, 6)")
print("="*65)

for row in unimmqa_rows:
    print(f"\n{'─'*65}")
    print(f"  [{row['type']}] {row['question']}")
    print(f"{'─'*65}")

    if row.get('ttext'):
        print("\n📊 MODULE 2 — Position-Enhanced Linearized Table:")
        print(row['ttext'][:300])

    if row.get('itext'):
        print("\n🖼️  MODULE 1 — Diversified Captions + OCR:")
        print(row['itext'][:300])

    if row.get('rationale'):
        print("\n🔗 MODULE 3 — Cross-Modal Rationale:")
        print(row['rationale'])

    print(f"\n💡 MODULE 4 — Answer:")
    print(f"   Raw:    {row['raw_answer']}")
    print(f"   Clean:  {row['prediction']}")
    print(f"   Gold:   {row['gold_answer']}")
    print(f"   EM: {'✅' if row['exact_match'] else '❌'}  F1: {row['token_f1']:.3f}")

print(f"\n{'='*65}")
print("  FINAL SUMMARY")
print(f"{'='*65}")
print(f"  QA Backbone: {QA_MODEL}")
print(f"  BLIP: N={N_CAPTIONS} captions | Top-K={TOP_K} | Top-p={TOP_P}")
print(f"  OCR: {USE_OCR} | Rationale: {USE_RATIONALE}")
print(f"  UniMMQA: EM={unimmqa_eval['EM%']}%  F1={unimmqa_eval['F1%']}%")
print(f"\n  Output files:")
for f in sorted(OUTPUT_DIR.glob("*.csv")) + sorted(OUTPUT_DIR.glob("*.jsonl")):
    print(f"    {f.name}  ({f.stat().st_size/1024:.1f} KB)")


  CASE STUDY — Module outputs for each example
  (Mirrors paper Figures 4, 5, 6)

─────────────────────────────────────────────────────────────────
  [TableQ] For which film did Ben Piazza play the role of Mr. Simms?
─────────────────────────────────────────────────────────────────

📊 MODULE 2 — Position-Enhanced Linearized Table:
table title : Ben Piazza
header : Year | Title | Role | Notes
row 1 : 1957 | A Dangerous Age | David | 
row 2 : 1959 | The Hanging Tree | Rune | 
row 3 : 1962 | No Exit | Camarero | 
row 4 : 1970 | Tell Me That You Love Me, Junie Moon | Jesse | 
row 5 : 1972 | The Outside Man | Desk Clerk | 
row 6 :

🖼️  MODULE 1 — Diversified Captions + OCR:
image: The Concorde ... Airport '79

🔗 MODULE 3 — Cross-Modal Rationale:
The Concorde ... Airport '79

💡 MODULE 4 — Answer:
   Raw:    MMQA
   Clean:  MMQA
   Gold:   Mask
   EM: ❌  F1: 0.000

─────────────────────────────────────────────────────────────────
  [Compose(TextQ,TableQ)] When was the movie that had Ben Piaz